# 3D CNN 

In [1]:
import tensorflow  as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from keras.layers import Input
from tensorflow.keras.layers import *
# inputs = keras.layers.Input(shape=(img_size[0], img_size[1], 1))
import pandas as pd
import numpy as np

# 1. LOAD DATA

In [6]:
# Open the file for reading
with open("tof.txt", "r") as drinking_event:
    lines = []  # This will store each line as a string
    for line in drinking_event:
        clean_line = line.strip()#
        #example data = '1746621089'
        # '145,144,146,145,146,144,144,147,155,153,157,158,156,155,153,153,167,167,164,166,166,165,164,164,177,181,178,179,180,177,178,177,189,192,192,193,192,190,190,190,203,204,206,205,201,203,205,202,218,217,214,214,215,209,208,205,201,195,192,190,188,188,192,200'
        lines.append(clean_line)#All lines from file
    frames_flat = []  
    timestamps = []   
    # Start from first line, step by 2 each time (0, 2, 4, ...)
    for i in range(0, len(lines)-1, 2):
        Timestamp = lines[i]       # This is the timestamp line
        data = lines[i+1]   # This is the sensor data line
        values = data.split(',')  # Split data at commas to get a list of strings
        if len(values) == 64:
            float_values = [int(v) for v in values]
            frames_flat.append(float_values) 
            timestamps.append(Timestamp)             
        else:
            print(f"Bad data at data line {i+2}: found {len(values)} values instead of 64")
frames_np = np.array(frames_flat)
frames_np = frames_np.reshape(-1, 8, 8)
n_frames = frames_np.shape[0]
print("\nSummary:")
print(f"Total valid frames loaded: {len(frames_flat)}")
print(f"Timestamps: {len(timestamps)}")


Summary:
Total valid frames loaded: 1174
Timestamps: 1174


In [7]:
import numpy as np

n_events = 10      # Number of random "sip" events
clip_len = 16      # Frames per event (minimum length for an event)
n_frames = 500     # Example value for total frames (replace with your real value!)

np.random.seed(0)  # For reproducible random numbers

# Randomly pick event start frames
all_possible_starts = np.arange(0, n_frames-clip_len, 30)
print("All possible start indices for events:", all_possible_starts)

event_starts = np.sort(np.random.choice(all_possible_starts, n_events, replace=False))
print("\nRandom event starts (sorted):", event_starts)

# Generate random end frames for each event
random_durations = np.random.randint(clip_len*2, clip_len*4, size=n_events)
print("\nRandom durations for each event:", random_durations)

event_ends = event_starts + random_durations
event_ends = np.minimum(event_ends, n_frames-1)   # Make sure no event goes beyond last frame!
print("\nCalculated event ends (after min check):", event_ends)

# Generate random drink volumes
event_volumes = np.round(np.random.uniform(20, 100, size=n_events), 1)
print("\nRandom event drink volumes (in mL):", event_volumes)

# Combine events into a final list of tuples
events = list(zip(event_starts, event_ends, event_volumes))
print("\nCombined event tuples (start, end, volume):")
for i, (start, end, vol) in enumerate(events):
    print(f"Event {i+1}: Start Frame = {start}, End Frame = {end}, Volume = {vol} mL")

All possible start indices for events: [  0  30  60  90 120 150 180 210 240 270 300 330 360 390 420 450 480]

Random event starts (sorted): [ 30  60 120 180 210 240 270 300 390 420]

Random durations for each event: [33 38 39 55 46 56 49 37 57 45]

Calculated event ends (after min check): [ 63  98 159 235 256 296 319 337 447 465]

Random event drink volumes (in mL): [47.  71.9 49.5 96.6 31.2 89.6 57.9 84.1 61.6 74.3]

Combined event tuples (start, end, volume):
Event 1: Start Frame = 30, End Frame = 63, Volume = 47.0 mL
Event 2: Start Frame = 60, End Frame = 98, Volume = 71.9 mL
Event 3: Start Frame = 120, End Frame = 159, Volume = 49.5 mL
Event 4: Start Frame = 180, End Frame = 235, Volume = 96.6 mL
Event 5: Start Frame = 210, End Frame = 256, Volume = 31.2 mL
Event 6: Start Frame = 240, End Frame = 296, Volume = 89.6 mL
Event 7: Start Frame = 270, End Frame = 319, Volume = 57.9 mL
Event 8: Start Frame = 300, End Frame = 337, Volume = 84.1 mL
Event 9: Start Frame = 390, End Frame = 44

In [8]:
import numpy as np

# Example data parameters
stride = 4           # Step size between each clip


# Prepare for slicing
clips = []
clip_start_idxs = []

print(f"Total frames: {n_frames}, Clip length: {clip_len}, Stride: {stride}\n")

# Sliding window slicing, with debug output
for start in range(0, n_frames - clip_len + 1, stride):
    end = start + clip_len
    print(f"Slicing frames {start} to {end-1} (clip indices: {start}:{end})")
    # Extract a clip from the data
    new_clip = frames_np[start:end]
    clips.append(new_clip)
    clip_start_idxs.append(start)

clips = np.array(clips)     # (n_clips, clip_len, 8, 8)
clip_start_idxs = np.array(clip_start_idxs)
clip_end_idxs = clip_start_idxs + clip_len - 1
n_clips = clips.shape[0]

print(f"\nNumber of clips: {n_clips}")
print("Clip start indices:", clip_start_idxs)
print("Clip end indices:", clip_end_idxs)
print("\nShape of clips array:", clips.shape)
print("First clip shape:", clips[0].shape)
print("First 2 frames of the first clip:\n", clips[0][:2])  # show first 2 frames in first clip

Total frames: 500, Clip length: 16, Stride: 4

Slicing frames 0 to 15 (clip indices: 0:16)
Slicing frames 4 to 19 (clip indices: 4:20)
Slicing frames 8 to 23 (clip indices: 8:24)
Slicing frames 12 to 27 (clip indices: 12:28)
Slicing frames 16 to 31 (clip indices: 16:32)
Slicing frames 20 to 35 (clip indices: 20:36)
Slicing frames 24 to 39 (clip indices: 24:40)
Slicing frames 28 to 43 (clip indices: 28:44)
Slicing frames 32 to 47 (clip indices: 32:48)
Slicing frames 36 to 51 (clip indices: 36:52)
Slicing frames 40 to 55 (clip indices: 40:56)
Slicing frames 44 to 59 (clip indices: 44:60)
Slicing frames 48 to 63 (clip indices: 48:64)
Slicing frames 52 to 67 (clip indices: 52:68)
Slicing frames 56 to 71 (clip indices: 56:72)
Slicing frames 60 to 75 (clip indices: 60:76)
Slicing frames 64 to 79 (clip indices: 64:80)
Slicing frames 68 to 83 (clip indices: 68:84)
Slicing frames 72 to 87 (clip indices: 72:88)
Slicing frames 76 to 91 (clip indices: 76:92)
Slicing frames 80 to 95 (clip indices: 

In [9]:
# labels = [0, 0, ..., 0] for all clips (default: not part of any sip event)
labels = np.zeros(n_clips, dtype=np.float32)   # n_clips = total number of clips
clips_labeled = 0

print(f"Total number of clips: {n_clips}")
print("\nLabeling each clip:")

for i in range(n_clips):
    labeled = False  # track if this clip was labeled as an event
    # For each event, check if this clip overlaps with the event window
    for ev in events:
        ev_start, ev_end, ev_vol = ev
        # Determine if the current clip (frames from clip_start_idxs[i] to clip_end_idxs[i]) overlaps the event
        overlap = (clip_end_idxs[i] >= ev_start) and (clip_start_idxs[i] <= ev_end)
        if overlap:
            labels[i] = ev_vol  # assign the event's volume to this clip
            labeled = True
            clips_labeled += 1
            print(f" Clip {i:3d} [{clip_start_idxs[i]}-{clip_end_idxs[i]}] overlaps event [{ev_start}-{ev_end}], volume={ev_vol} mL")
            break  # Only use the first (earliest) event that overlaps
    if not labeled:
        print(f" Clip {i:3d} [{clip_start_idxs[i]}-{clip_end_idxs[i]}] has no event (label=0)")

print(f"\nTotal clips labeled as 'sip event': {clips_labeled} ({(clips_labeled / n_clips) * 100:.1f}% of all clips)")
print("Sample of final clip labels:", labels[:20])

Total number of clips: 122

Labeling each clip:
 Clip   0 [0-15] has no event (label=0)
 Clip   1 [4-19] has no event (label=0)
 Clip   2 [8-23] has no event (label=0)
 Clip   3 [12-27] has no event (label=0)
 Clip   4 [16-31] overlaps event [30-63], volume=47.0 mL
 Clip   5 [20-35] overlaps event [30-63], volume=47.0 mL
 Clip   6 [24-39] overlaps event [30-63], volume=47.0 mL
 Clip   7 [28-43] overlaps event [30-63], volume=47.0 mL
 Clip   8 [32-47] overlaps event [30-63], volume=47.0 mL
 Clip   9 [36-51] overlaps event [30-63], volume=47.0 mL
 Clip  10 [40-55] overlaps event [30-63], volume=47.0 mL
 Clip  11 [44-59] overlaps event [30-63], volume=47.0 mL
 Clip  12 [48-63] overlaps event [30-63], volume=47.0 mL
 Clip  13 [52-67] overlaps event [30-63], volume=47.0 mL
 Clip  14 [56-71] overlaps event [30-63], volume=47.0 mL
 Clip  15 [60-75] overlaps event [30-63], volume=47.0 mL
 Clip  16 [64-79] overlaps event [60-98], volume=71.9 mL
 Clip  17 [68-83] overlaps event [60-98], volume=7

# hyperpaprameter

In [ ]:
clip_length = 16 
height, width =8,8
channels= 1



# Model definition 

In [ ]:
def make_gesture_model():
    model= models.Sequential([
        layers.input(shape= (clip_length, height, width, channels)),
        layers.Conv3D(8,kernel_size=(3,3,3),activation='relu',padding='same'),
        layers.BatchNormalization(),
        layers.Conv3D(16,kernel_size=(3,3,3),activation='relu', strides=(1,2,2),padding='same'),
        layers.BatchNormalzation(),
        layers.GlobalAverragePooling3D(),
        layers.Dense(32,activation='relu'),
        layers.Dense(1,activation='sigmoid')
    ])
    return model

model = make_gesture_model()
model.compile(optimizer ='adam',
              loss= 'binary_crossentropy',
              metrics=['accuracy'])
model.summary()
    

# training

In [ ]:
history= model.fit(X_train,y_train,batch_size=128,epochs=30, validatio_data(X_val,y_val),class_weight={0:1.0,1: 2.0})


# testing

In [ ]:
proba = model.predict(new_clip)
is_drinking = proba[0,0] > 0.5


# extact_sips

In [ ]:
def extract_sips(preds, threshold=0.5, min_frame=5, stride=1):
    sipping = preds > threshold
    sips= []
    sip_start = None
    for i, state in enumerate(sipping):
        if state and sip_start is None:
            sip_start = i
        elif not state and sip_start is not None:
            if (i - sip_start) * stride >= min_frames:
                sips.append((sip_start * stride, i* stride))
            sip_start = None
    if sip_start is not None:
        sips.append((sip_start * stride, len(sipping) * stride))
    return sips

In [ ]:
import numpy as np

# 1. LOAD DATA (comma-separated)
# --------------------------------
# with open("tof.txt", "r") as f:
#     frames_flat = [list(map(float, line.strip().split(','))) for line in f.readlines()]
# frames_np = np.array(frames_flat)         # shape (n_frames, 64)
# frames_np = frames_np.reshape(-1, 8, 8)   # shape (n_frames, 8, 8)
# n_frames = frames_np.shape[0]
# frames_flat = []
# timestamps = []
with open("tof.txt", "r") as f:
    lines = [line.strip() for line in f]
    for i in range(0, len(lines)-1, 2):  # Step by 2
        ts = lines[i]
        data = lines[i+1]
        values = data.split(',')
        if len(values) == 64:
            frames_flat.append([float(v) for v in values])
            timestamps.append(ts)
        else:
            print(f"Bad data at line {i+2} with {len(values)} values.")

frames_np = np.array(frames_flat)
frames_np = frames_np.reshape(-1, 8, 8)
n_frames = frames_np.shape[0]
print(f"Good frames loaded: {n_frames}")
# 2. SIMULATE RANDOM DRINKING EVENTS & RANDOM VOLUMES
# ----------------------------------------------------
n_events = 10   # number of random "sip" events
clip_len = 16   # frames per clip
np.random.seed(0)
event_starts = np.sort(np.random.choice(np.arange(0, n_frames-clip_len, 30), n_events, replace=False))
event_ends = event_starts + np.random.randint(clip_len*2, clip_len*4, size=n_events)
event_ends = np.minimum(event_ends, n_frames-1)
event_volumes = np.round(np.random.uniform(20, 100, size=n_events), 1)   # mL

events = list(zip(event_starts, event_ends, event_volumes))

# 3. SLICE DATA INTO CLIPS
# ------------------------
stride = 4
clips = []
clip_start_idxs = []
for start in range(0, n_frames - clip_len + 1, stride):
    clips.append(frames_np[start:start+clip_len])
    clip_start_idxs.append(start)
clips = np.array(clips)     # (n_clips, clip_len, 8, 8)
clip_start_idxs = np.array(clip_start_idxs)
clip_end_idxs = clip_start_idxs + clip_len - 1
n_clips = clips.shape[0]

# 4. LABEL EACH CLIP WITH VOLUME (regression target)
# --------------------------------------------------
labels = np.zeros(n_clips, dtype=np.float32)   # default: 0 mL (no sip)
for i in range(n_clips):
    for ev_start, ev_end, ev_vol in events:
        overlap = (clip_end_idxs[i] >= ev_start) and (clip_start_idxs[i] <= ev_end)
        if overlap:
            labels[i] = ev_vol
            break

# 5. PREPARE DATA FOR ML MODEL
# ----------------------------
# Model expects (batch, time, height, width, channel)
clips = clips[..., None]  # add channel dim: shape (n_clips, 16, 8, 8, 1)

# Train/valid split (90/10)
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(clips, labels, test_size=0.1, random_state=42)

# 6. BUILD & TRAIN SIMPLE KERAS REGRESSION MODEL
# ----------------------------------------------
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Simple 3D CNN model for regression
model = keras.Sequential([
    layers.Input(shape=(clip_len, 8, 8, 1)),
    layers.Conv3D(16, (3, 3, 3), activation='relu', padding='same'),
    layers.MaxPooling3D((2, 2, 2)),
    layers.Conv3D(32, (3, 3, 3), activation='relu', padding='same'),
    layers.MaxPooling3D((2, 2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # regression: output=volume in mL
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# 7. TRAIN
# ---------
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=5, batch_size=16)

# 8. PRINT SAMPLE PREDICTIONS
# ---------------------------
y_pred = model.predict(X_val[:10]).flatten()
print("\nSample predictions (ml):")
for i in range(10):
    print(f" True: {y_val[i]:6.1f} | Predicted: {y_pred[i]:6.1f}")

# Done: This code shows the whole process
